In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in working directory)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric
for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# Helper: compute min/max epoch and localized datetime strings
def ts_bounds(df):
    if df.shape[0] == 0:
        return (None, None, None, None)
    min_epoch = int(df['timestamp'].min())
    max_epoch = int(df['timestamp'].max())
    # convert to tz-aware datetimes
    min_dt = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz)
    max_dt = pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz)
    return (min_epoch, min_dt.isoformat(), max_epoch, max_dt.isoformat())

# Incident window: March 10, 2021 08:30 to 09:00 (UTC+8)
start_local = tz.localize(datetime(2021, 3, 10, 8, 30, 0))
end_local = tz.localize(datetime(2021, 3, 10, 9, 0, 0))
# convert to UTC epoch seconds for comparison with timestamp column
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Summaries for each file
metric_min_epoch, metric_min_dt, metric_max_epoch, metric_max_dt = ts_bounds(metric)
trace_min_epoch, trace_min_dt, trace_max_epoch, trace_max_dt = ts_bounds(trace)
log_min_epoch, log_min_dt, log_max_epoch, log_max_dt = ts_bounds(log)
err_min_epoch, err_min_dt, err_max_epoch, err_max_dt = ts_bounds(error_logs)

# Unique cmdb_id lists (up to 20)
metric_cmdbs = metric['cmdb_id'].dropna().drop_duplicates().sort_values().head(20)
trace_cmdbs = trace['cmdb_id'].dropna().drop_duplicates().sort_values().head(20)
log_cmdbs = log['cmdb_id'].dropna().drop_duplicates().sort_values().head(20)
err_cmdbs = error_logs['cmdb_id'].dropna().drop_duplicates().sort_values().head(20)

# Top 20 KPI/trace/log/message counts
metric_kpi_counts = metric.groupby('kpi_name').size().sort_values(ascending=False).reset_index(name='count').head(20)
trace_name_counts = trace.groupby('trace_name').size().sort_values(ascending=False).reset_index(name='count').head(20)
log_name_counts = log.groupby('log_name').size().sort_values(ascending=False).reset_index(name='count').head(20)
error_msg_counts = error_logs.groupby('message').size().sort_values(ascending=False).reset_index(name='count').head(20)

# Incident window row counts per file (inclusive)
metric_incident_count = metric[(metric['timestamp'] >= start_epoch) & (metric['timestamp'] <= end_epoch)].shape[0]
trace_incident_count = trace[(trace['timestamp'] >= start_epoch) & (trace['timestamp'] <= end_epoch)].shape[0]
log_incident_count = log[(log['timestamp'] >= start_epoch) & (log['timestamp'] <= end_epoch)].shape[0]
error_incident_count = error_logs[(error_logs['timestamp'] >= start_epoch) & (error_logs['timestamp'] <= end_epoch)].shape[0]

# Compact summary DataFrames (one-row each)
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric.shape[0],
    'min_ts_epoch': metric_min_epoch,
    'min_ts_local': metric_min_dt,
    'max_ts_epoch': metric_max_epoch,
    'max_ts_local': metric_max_dt,
    'unique_cmdb_count': metric['cmdb_id'].nunique(dropna=True),
    'incident_rows_2021-03-10_08:30-09:00+8': metric_incident_count
}])

trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace.shape[0],
    'min_ts_epoch': trace_min_epoch,
    'min_ts_local': trace_min_dt,
    'max_ts_epoch': trace_max_epoch,
    'max_ts_local': trace_max_dt,
    'unique_cmdb_count': trace['cmdb_id'].nunique(dropna=True),
    'incident_rows_2021-03-10_08:30-09:00+8': trace_incident_count
}])

log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'rows': log.shape[0],
    'min_ts_epoch': log_min_epoch,
    'min_ts_local': log_min_dt,
    'max_ts_epoch': log_max_epoch,
    'max_ts_local': log_max_dt,
    'unique_cmdb_count': log['cmdb_id'].nunique(dropna=True),
    'incident_rows_2021-03-10_08:30-09:00+8': log_incident_count
}])

error_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'rows': error_logs.shape[0],
    'min_ts_epoch': err_min_epoch,
    'min_ts_local': err_min_dt,
    'max_ts_epoch': err_max_epoch,
    'max_ts_local': err_max_dt,
    'unique_cmdb_count': error_logs['cmdb_id'].nunique(dropna=True),
    'incident_rows_2021-03-10_08:30-09:00+8': error_incident_count
}])

# Final displayed compact outputs (only aggregates/top lists; keep compact)
metric_summary, metric_kpi_counts, metric_cmdbs, trace_summary, trace_name_counts, trace_cmdbs, log_summary, log_name_counts, log_cmdbs, error_summary, error_msg_counts, err_cmdbs

```
Out[1]:
```
Summary (incident window = 2021-03-10 08:30:00 to 2021-03-10 09:00:00, timezone UTC+8)

metric.csv
- Total rows: 28,436
- Timestamp range (local): 2021-03-10T08:30:00+08:00 — 2021-03-10T09:00:00+08:00
- Unique cmdb_id (up to 20 shown): IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, ServiceTest1, ServiceTest10, ServiceTest11, ServiceTest2, ServiceTest3, ServiceTest4, ServiceTest5, ServiceTest6, ServiceTest7, ServiceTest8, ServiceTest9, Tomcat01
- Top kpi_name (by count, top 20):
  1. OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — 368
  2. OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount — 367
  3. OSLinux-OSLinux_ZABBIX_Host_Uptime — 367
  4. OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — 367
  5. OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — 365
  6. OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_ProcessCount — 360
  7. OSLinux-CPU_CPU_CPUWio — 360
  8. OSLinux-CPU_CPU_CPUSysTime — 360
  9. OSLinux-CPU_CPU_CPUUserTime — 360
 10. OSLinux-CPU_CPU_CPUidleutil — 360
 11. OSLinux-CPU_CPU_CPULoad — 360
 12. app.sr — 341
 13. app.cnt — 341
 14. app.mrt — 341
 15. app.rr — 341
 16. OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc — 299
 17. OSLinux-CPU_CPU_CPUCpuUtil — 288
 18. OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc — 287
 19. OSLinux-OSLinux_MEMORY_MEMORY_UserMem — 285
 20. OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem — 277
- Rows inside incident window: 28,436 (all rows fall within the specified window)

trace.csv
- Total rows: 6,879
- Timestamp range (local): 2021-03-10T08:30:00+08:00 — 2021-03-10T08:59:00+08:00
- Unique cmdb_id (up to 20 shown): IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, Tomcat03, Tomcat04, dockerA1, dockerA2, dockerB1, dockerB2
- Top trace_name (by count, top 20):
  1. trace.self.duration_p95 — 360
  2. trace.self.row_count — 360
  3. trace.self.duration_mean — 360
  4. trace.to_MG01.row_count — 240
  5. trace.to_MG02.duration_p95 — 240
  6. trace.to_MG02.duration_mean — 240
  7. trace.to_MG01.duration_mean — 240
  8. trace.to_MG01.duration_p95 — 240
  9. trace.to_MG02.row_count — 240
 10. trace.from_MG01.duration_mean — 120
 11. trace.from_IG02.row_count — 120
 12. trace.from_IG02.duration_p95 — 120
 13. trace.from_IG02.duration_mean — 120
 14. trace.from_IG01.row_count — 120
 15. trace.from_IG01.duration_p95 — 120
 16. trace.from_IG01.duration_mean — 120
 17. trace.from_MG01.duration_p95 — 120
 18. trace.from_MG01.row_count — 120
 19. trace.from_MG02.duration_p95 — 120
 20. trace.from_MG02.duration_mean — 120
- Rows inside incident window: 6,879 (all rows fall within the specified window)

log.csv
- Total rows: 469
- Timestamp range (local): 2021-03-10T08:30:00+08:00 — 2021-03-10T09:00:00+08:00
- Unique cmdb_id (up to 20 shown): Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02
- Top log_name (by count, top 20; available top 3 shown):
  1. log.error_count — 205
  2. log.row_count — 205
  3. log.gc_pause_max — 59
- Rows inside incident window: 469 (all rows fall within the specified window)

error_logs.csv
- Total rows: 0
- Timestamp range: none (file is empty)
- Unique cmdb_id: none
- Top message values: none (no entries)
- Rows inside incident window: 0

Notes:
- All files with data (metric, trace, log) have rows that fall entirely within the specified incident window.
- error_logs.csv contains no records.

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows  min_ts_epoch               min_ts_local  max_ts_epoch               max_ts_local  unique_cmdb_count  incident_rows_2021-03-10_08:30-09:00+8
0  metric.csv  28436    1615336200  2021-03-10T08:30:00+08:00    1615338000  2021-03-10T09:00:00+08:00                 29                                   28436,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    368
1          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    367
2                  OSLinux-OSLinux_ZABBIX_Host_Uptime    367
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    367
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    365
5   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    360
6                              OSLinux-CPU_CPU_CPUWio    360
7                          OSLinux-CPU_CPU_CPUSysTime    360
8                         OSLinux-CPU_CPU_CPUUserTime    360
9                         OSLinux-CPU_CPU_CPUidleutil    360
10                            OSLinux-CPU_CPU_CPULoad    360
11                                             app.sr    341
12                                            app.cnt    341
13                                            app.mrt    341
14                                             app.rr    341
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    299
16                         OSLinux-CPU_CPU_CPUCpuUtil    288
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    287
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    285
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    277, 0               IG01
60              IG02
128             MG01
173             MG02
230          Mysql01
381          Mysql02
580          Redis01
660          Redis02
723     ServiceTest1
727    ServiceTest10
731    ServiceTest11
735     ServiceTest2
739     ServiceTest3
743     ServiceTest4
747     ServiceTest5
751     ServiceTest6
755     ServiceTest7
759     ServiceTest8
763     ServiceTest9
767         Tomcat01
Name: cmdb_id, dtype: object,         file  rows  min_ts_epoch               min_ts_local  max_ts_epoch               max_ts_local  unique_cmdb_count  incident_rows_2021-03-10_08:30-09:00+8
0  trace.csv  6879    1615336200  2021-03-10T08:30:00+08:00    1615337940  2021-03-10T08:59:00+08:00                 12                                    6879,                        trace_name  count
0         trace.self.duration_p95    360
1            trace.self.row_count    360
2        trace.self.duration_mean    360
3         trace.to_MG01.row_count    240
4      trace.to_MG02.duration_p95    240
5     trace.to_MG02.duration_mean    240
6     trace.to_MG01.duration_mean    240
7      trace.to_MG01.duration_p95    240
8         trace.to_MG02.row_count    240
9   trace.from_MG01.duration_mean    120
10      trace.from_IG02.row_count    120
11   trace.from_IG02.duration_p95    120
12  trace.from_IG02.duration_mean    120
13      trace.from_IG01.row_count    120
14   trace.from_IG01.duration_p95    120
15  trace.from_IG01.duration_mean    120
16   trace.from_MG01.duration_p95    120
17      trace.from_MG01.row_count    120
18   trace.from_MG02.duration_p95    120
19  trace.from_MG02.duration_mean    120, 0          IG01
15         IG02
30         MG01
69         MG02
108    Tomcat01
123    Tomcat02
141    Tomcat03
156    Tomcat04
171    dockerA1
186    dockerA2
201    dockerB1
216    dockerB2
Name: cmdb_id, dtype: object,       file  rows  min_ts_epoch               min_ts_local  max_ts_epoch               max_ts_local  unique_cmdb_count  incident_rows_2021-03-10_08:30-09:00+8
0  log.csv   469    1615336200  2021-03-10T08:30:00+08:00    1615338000  2021-03-10T09:00:00+08:00                  7                                     469,            log_name  count
0   log.error_count    205
1     log.row_count    205
2  log.gc_pause_max     59, 0      Mysql01
2     Tomcat01
4     Tomcat02
7     Tomcat03
9     Tomcat04
11    apache01
13    apache02
Name: cmdb_id, dtype: object,              file  rows min_ts_epoch min_ts_local max_ts_epoch max_ts_local  unique_cmdb_count  incident_rows_2021-03-10_08:30-09:00+8
0  error_logs.csv     0         None         None         None         None                  0                                       0, Empty DataFrame
Columns: [message, count]
Index: [], Series([], Name: cmdb_id, dtype: object))```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing variables: metric, start_epoch, end_epoch, tz
# define allowed components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Filter metric to allowed cmdb_ids
metric_allowed = metric[metric['cmdb_id'].isin(allowed)].copy()

# Ensure numeric value
metric_allowed['value'] = pd.to_numeric(metric_allowed['value'], errors='coerce')

# Global summary per (cmdb_id, kpi_name): count, min, max, p95, p5
grp = metric_allowed.groupby(['cmdb_id','kpi_name'])['value']
global_summary = grp.agg(
    count_global='count',
    min_global='min',
    max_global='max',
    p95=lambda s: s.quantile(0.95),
    p5=lambda s: s.quantile(0.05)
).reset_index()

# Filter to incident window (inclusive)
metric_window = metric_allowed[(metric_allowed['timestamp'] >= start_epoch) & (metric_allowed['timestamp'] <= end_epoch)].copy()

# Merge global thresholds into window rows to compare
window_merged = metric_window.merge(global_summary, on=['cmdb_id','kpi_name'], how='left')

# Flags for anomalies vs global thresholds
window_merged['gt_p95'] = window_merged['value'] > window_merged['p95']
window_merged['lt_p5'] = window_merged['value'] < window_merged['p5']

# Aggregate window-level stats for groups that have points in window
def agg_window(g):
    pts = int(g.shape[0])
    num_gt = int(g['gt_p95'].sum())
    earliest_gt = int(g.loc[g['gt_p95'], 'timestamp'].min()) if num_gt>0 else pd.NA
    max_inside = g['value'].max() if pts>0 else pd.NA
    num_lt = int(g['lt_p5'].sum())
    earliest_lt = int(g.loc[g['lt_p5'], 'timestamp'].min()) if num_lt>0 else pd.NA
    min_inside = g['value'].min() if pts>0 else pd.NA
    return pd.Series({
        'points_inside': pts,
        'num_gt_p95': num_gt,
        'earliest_gt_p95_epoch': earliest_gt,
        'max_inside': max_inside,
        'num_lt_p5': num_lt,
        'earliest_lt_p5_epoch': earliest_lt,
        'min_inside': min_inside
    })

if not window_merged.empty:
    window_stats = window_merged.groupby(['cmdb_id','kpi_name']).apply(agg_window).reset_index()
else:
    # empty frame with expected columns
    window_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','points_inside','num_gt_p95',
                                         'earliest_gt_p95_epoch','max_inside','num_lt_p5',
                                         'earliest_lt_p5_epoch','min_inside'])

# Left-join global_summary with window_stats so pairs with zero points in window are kept
full_agg = global_summary.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for numeric counts with 0 and keep epoch earliest as NA where no events
num_cols = ['points_inside','num_gt_p95','max_inside','num_lt_p5','min_inside']
for c in num_cols:
    if c in full_agg.columns:
        full_agg[c] = full_agg[c].fillna(0)

# Convert earliest epoch timestamps to localized ISO strings (UTC+8) for readability
def epoch_to_local_iso(val):
    if pd.isna(val):
        return pd.NA
    try:
        return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()
    except Exception:
        return pd.NA

full_agg['earliest_gt_p95_local'] = full_agg['earliest_gt_p95_epoch'].apply(epoch_to_local_iso)
full_agg['earliest_lt_p5_local'] = full_agg['earliest_lt_p5_epoch'].apply(epoch_to_local_iso)

# Compute total anomalies and a simple severity score (num anomalies)
full_agg['total_anomalies'] = full_agg['num_gt_p95'].astype(int) + full_agg['num_lt_p5'].astype(int)

# Sort by total_anomalies desc, then by points_inside desc, keep top 20 for compact display
top20_summary = full_agg.sort_values(['total_anomalies','points_inside'], ascending=[False, False]).head(20)

# Select and order compact columns to display
display_cols = [
    'cmdb_id','kpi_name',
    'count_global','min_global','max_global','p95','p5',
    'points_inside','num_gt_p95','earliest_gt_p95_local','max_inside',
    'num_lt_p5','earliest_lt_p5_local','min_inside',
    'total_anomalies'
]
# Ensure columns exist
display_cols = [c for c in display_cols if c in top20_summary.columns]

# Store full aggregated summaries in kernel variables (as required)
global_summary_full = global_summary        # global per-pair thresholds and stats
window_stats_full = window_stats            # full window-level aggregated stats
full_aggregated_table = full_agg            # merged full table (global + window stats)

# Display the compact top-20 summary (as required, variable output)
top20_summary[display_cols].head(20)

```
Out[2]:
```
Summary of results (metric rows filtered to allowed components; incident window 2021-03-10 08:30–09:00 UTC+8):

What I did
- Kept only allowed components and computed global statistics per (cmdb_id, kpi_name): global count, min, max, P95, P5.
- Then restricted rows to the incident window and computed per-pair window stats: points inside window, count of values > global P95, earliest timestamp (local) of >P95 if any, max inside window, count of values < global P5, earliest timestamp (local) of <P5 if any, min inside window.
- Ranked pairs by total anomaly count (points >P95 + points <P5). Top 20 pairs are shown below. Full aggregated tables are stored in kernel variables global_summary_full, window_stats_full, and full_aggregated_table.

Top 20 (cmdb_id — kpi_name): key stats (global count; points inside; >P95 count & earliest; max inside; <P5 count & earliest; min inside; total anomalies)

1. IG01 — JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:44:00+08:00); max_inside 8.41212e+08
   - <P5: 2 (first 2021-03-10T08:30:00+08:00); min_inside 2.795594e+07
   - total_anomalies: 4

2. IG01 — OSLinux-CPU_CPU_CPULoad
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:42:00+08:00); max_inside 1.2
   - <P5: 2 (first 2021-03-10T08:38:00+08:00); min_inside 1.0
   - total_anomalies: 4

3. IG01 — OSLinux-CPU_CPU_CPUSysTime
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:39:00+08:00); max_inside 0.5281
   - <P5: 2 (first 2021-03-10T08:32:00+08:00); min_inside 0.2994
   - total_anomalies: 4

4. IG01 — OSLinux-CPU_CPU_CPUUserTime
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:39:00+08:00); max_inside 25.3244
   - <P5: 2 (first 2021-03-10T08:32:00+08:00); min_inside 25.16942
   - total_anomalies: 4

5. IG01 — OSLinux-CPU_CPU_CPUWio
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:47:00+08:00); max_inside 0.4865
   - <P5: 2 (first 2021-03-10T08:54:00+08:00); min_inside 0.0125
   - total_anomalies: 4

6. IG01 — OSLinux-CPU_CPU_CPUidleutil
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:31:00+08:00); max_inside 74.3067
   - <P5: 2 (first 2021-03-10T08:53:00+08:00); min_inside 73.7173
   - total_anomalies: 4

7. IG01 — OSLinux-OSLinux_ZABBIX_Host_Uptime
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:59:00+08:00); max_inside 6.537007e+06
   - <P5: 2 (first 2021-03-10T08:30:00+08:00); min_inside 6.5352e+06
   - total_anomalies: 4

8. IG02 — JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:36:00+08:00); max_inside 8.255171e+08
   - <P5: 2 (first 2021-03-10T08:37:00+08:00); min_inside 3.490951e+07
   - total_anomalies: 4

9. IG02 — OSLinux-CPU_CPU_CPUUserTime
   - global count 31; points_inside 31
   - >P95: 2 (first 2021-03-10T08:44:00+08:00); max_inside 25.40105
   - <P5: 2 (first 2021-03-10T08:33:00+08:00); min_inside 25.19122
   - total_anomalies: 4

10. IG02 — OSLinux-CPU_CPU_CPUWio
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:56:00+08:00); max_inside 0.810676
    - <P5: 2 (first 2021-03-10T08:34:00+08:00); min_inside 0.0166
    - total_anomalies: 4

11. IG02 — OSLinux-CPU_CPU_CPUidleutil
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:31:00+08:00); max_inside 74.2708
    - <P5: 2 (first 2021-03-10T08:56:00+08:00); min_inside 73.21028
    - total_anomalies: 4

12. IG02 — OSLinux-OSLinux_ZABBIX_Host_Uptime
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:59:00+08:00); max_inside 6.537060e+06
    - <P5: 2 (first 2021-03-10T08:30:00+08:00); min_inside 6.535254e+06
    - total_anomalies: 4

13. MG01 — JVM-Memory_7779_JVM_Memory_HeapMemoryUsed
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:34:00+08:00); max_inside 1.060421e+09
    - <P5: 2 (first 2021-03-10T08:43:00+08:00); min_inside 2.256574e+08
    - total_anomalies: 4

14. MG02 — JVM-Memory_7779_JVM_Memory_HeapMemoryUsed
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:32:00+08:00); max_inside 1.195933e+09
    - <P5: 2 (first 2021-03-10T08:42:00+08:00); min_inside 3.973913e+08
    - total_anomalies: 4

15. Mysql01 — OSLinux-CPU_CPU_CPUSysTime
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:53:00+08:00); max_inside 0.565207
    - <P5: 2 (first 2021-03-10T08:32:00+08:00); min_inside 0.2411
    - total_anomalies: 4

16. Mysql01 — OSLinux-CPU_CPU_CPUUserTime
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:58:00+08:00); max_inside 0.4239
    - <P5: 2 (first 2021-03-10T08:30:00+08:00); min_inside 0.23278
    - total_anomalies: 4

17. Mysql01 — OSLinux-CPU_CPU_CPUWio
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:43:00+08:00); max_inside 2.8233
    - <P5: 2 (first 2021-03-10T08:37:00+08:00); min_inside 0.3948
    - total_anomalies: 4

18. Mysql01 — OSLinux-CPU_CPU_CPUidleutil
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:50:00+08:00); max_inside 98.9275
    - <P5: 2 (first 2021-03-10T08:43:00+08:00); min_inside 96.3659
    - total_anomalies: 4

19. Mysql01 — OSLinux-OSLinux_ZABBIX_Host_Uptime
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:59:00+08:00); max_inside 6.536613e+06
    - <P5: 2 (first 2021-03-10T08:30:00+08:00); min_inside 6.534806e+06
    - total_anomalies: 4

20. Mysql02 — OSLinux-CPU_CPU_CPULoad
    - global count 31; points_inside 31
    - >P95: 2 (first 2021-03-10T08:32:00+08:00); max_inside 1.22
    - <P5: 2 (first 2021-03-10T08:43:00+08:00); min_inside 0.4
    - total_anomalies: 4

Key takeaway
- For the top-20 (cmdb_id, kpi_name) pairs shown, each has the full global sample (31 points) and all 31 points fall inside the incident window. Each pair shows 2 points above its global P95 and 2 points below its global P5, yielding total_anomalies = 4 for these top pairs. Full aggregated results are available in kernel variables global_summary_full, window_stats_full, and full_aggregated_table if you want to inspect all pairs or export further.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   kpi_name  count_global    min_global    max_global           p95            p5  points_inside  num_gt_p95      earliest_gt_p95_local    max_inside  num_lt_p5       earliest_lt_p5_local    min_inside  total_anomalies
2       IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  2.795594e+07  8.412120e+08  8.047046e+08  5.449960e+07           31.0         2.0  2021-03-10T08:44:00+08:00  8.412120e+08        2.0  2021-03-10T08:30:00+08:00  2.795594e+07                4
12      IG01                    OSLinux-CPU_CPU_CPULoad            31  1.000000e+00  1.200000e+00  1.185000e+00  1.005000e+00           31.0         2.0  2021-03-10T08:42:00+08:00  1.200000e+00        2.0  2021-03-10T08:38:00+08:00  1.000000e+00                4
13      IG01                 OSLinux-CPU_CPU_CPUSysTime            31  2.994000e-01  5.281000e-01  4.803125e-01  3.035090e-01           31.0         2.0  2021-03-10T08:39:00+08:00  5.281000e-01        2.0  2021-03-10T08:32:00+08:00  2.994000e-01                4
14      IG01                OSLinux-CPU_CPU_CPUUserTime            31  2.516942e+01  2.532440e+01  2.529630e+01  2.517410e+01           31.0         2.0  2021-03-10T08:39:00+08:00  2.532440e+01        2.0  2021-03-10T08:32:00+08:00  2.516942e+01                4
15      IG01                     OSLinux-CPU_CPU_CPUWio            31  1.250000e-02  4.865000e-01  3.326500e-01  1.455000e-02           31.0         2.0  2021-03-10T08:47:00+08:00  4.865000e-01        2.0  2021-03-10T08:54:00+08:00  1.250000e-02                4
16      IG01                OSLinux-CPU_CPU_CPUidleutil            31  7.371730e+01  7.430670e+01  7.429314e+01  7.386524e+01           31.0         2.0  2021-03-10T08:31:00+08:00  7.430670e+01        2.0  2021-03-10T08:53:00+08:00  7.371730e+01                4
89      IG01         OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.535200e+06  6.537007e+06  6.536916e+06  6.535290e+06           31.0         2.0  2021-03-10T08:59:00+08:00  6.537007e+06        2.0  2021-03-10T08:30:00+08:00  6.535200e+06                4
92      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  3.490951e+07  8.255171e+08  7.822595e+08  6.013061e+07           31.0         2.0  2021-03-10T08:36:00+08:00  8.255171e+08        2.0  2021-03-10T08:37:00+08:00  3.490951e+07                4
104     IG02                OSLinux-CPU_CPU_CPUUserTime            31  2.519122e+01  2.540105e+01  2.534130e+01  2.520105e+01           31.0         2.0  2021-03-10T08:44:00+08:00  2.540105e+01        2.0  2021-03-10T08:33:00+08:00  2.519122e+01                4
105     IG02                     OSLinux-CPU_CPU_CPUWio            31  1.660000e-02  8.106760e-01  2.722950e-01  1.871400e-02           31.0         2.0  2021-03-10T08:56:00+08:00  8.106760e-01        2.0  2021-03-10T08:34:00+08:00  1.660000e-02                4
106     IG02                OSLinux-CPU_CPU_CPUidleutil            31  7.321028e+01  7.427080e+01  7.425128e+01  7.377325e+01           31.0         2.0  2021-03-10T08:31:00+08:00  7.427080e+01        2.0  2021-03-10T08:56:00+08:00  7.321028e+01                4
178     IG02         OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.535254e+06  6.537060e+06  6.536970e+06  6.535345e+06           31.0         2.0  2021-03-10T08:59:00+08:00  6.537060e+06        2.0  2021-03-10T08:30:00+08:00  6.535254e+06                4
181     MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  2.256574e+08  1.060421e+09  1.023540e+09  2.615987e+08           31.0         2.0  2021-03-10T08:34:00+08:00  1.060421e+09        2.0  2021-03-10T08:43:00+08:00  2.256574e+08                4
270     MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  3.973913e+08  1.195933e+09  1.064178e+09  3.978340e+08           31.0         2.0  2021-03-10T08:32:00+08:00  1.195933e+09        2.0  2021-03-10T08:42:00+08:00  3.973913e+08                4
494  Mysql01                 OSLinux-CPU_CPU_CPUSysTime            31  2.411000e-01  5.652070e-01  5.111955e-01  2.701345e-01           31.0         2.0  2021-03-10T08:53:00+08:00  5.652070e-01        2.0  2021-03-10T08:32:00+08:00  2.411000e-01                4
495  Mysql01                OSLinux-CPU_CPU_CPUUserTime            31  2.327800e-01  4.239000e-01  3.907010e-01  2.514675e-01           31.0         2.0  2021-03-10T08:58:00+08:00  4.239000e-01        2.0  2021-03-10T08:30:00+08:00  2.327800e-01                4
496  Mysql01                     OSLinux-CPU_CPU_CPUWio            31  3.948000e-01  2.823300e+00  2.040531e+00  5.071465e-01           31.0         2.0  2021-03-10T08:43:00+08:00  2.823300e+00        2.0  2021-03-10T08:37:00+08:00  3.948000e-01                4
497  Mysql01                OSLinux-CPU_CPU_CPUidleutil            31  9.636590e+01  9.892750e+01  9.888820e+01  9.710962e+01           31.0         2.0  2021-03-10T08:50:00+08:00  9.892750e+01        2.0  2021-03-10T08:43:00+08:00  9.636590e+01                4
584  Mysql01         OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.534806e+06  6.536613e+06  6.536522e+06  6.534896e+06           31.0         2.0  2021-03-10T08:59:00+08:00  6.536613e+06        2.0  2021-03-10T08:30:00+08:00  6.534806e+06                4
718  Mysql02                    OSLinux-CPU_CPU_CPULoad            31  4.000000e-01  1.220000e+00  1.160000e+00  5.200000e-01           31.0         2.0  2021-03-10T08:32:00+08:00  1.220000e+00        2.0  2021-03-10T08:43:00+08:00  4.000000e-01                4```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse variables from previous steps:
# metric_allowed, window_merged, global_summary_full, tz, start_epoch, end_epoch

# Ensure window_merged exists and has necessary columns; if not, rebuild quickly from metric_allowed and global_summary_full
if 'window_merged' not in globals():
    metric_window = metric_allowed[(metric_allowed['timestamp'] >= start_epoch) & (metric_allowed['timestamp'] <= end_epoch)].copy()
    window_merged = metric_window.merge(global_summary_full, on=['cmdb_id','kpi_name'], how='left')
else:
    # ensure numeric
    window_merged['value'] = pd.to_numeric(window_merged['value'], errors='coerce')

# Flag anomalies (high / low)
window_merged['gt_p95'] = window_merged['value'] > window_merged['p95']
window_merged['lt_p5'] = window_merged['value'] < window_merged['p5']
# Keep only anomalous points for segment detection
anomaly_df = window_merged[window_merged['gt_p95'] | window_merged['lt_p5']].copy()

# If no anomalies, create empty result
if anomaly_df.empty:
    metric_fault_segments = pd.DataFrame(columns=[
        'component','kpi_name','fault_start_local','fault_end_local','length_in_points',
        'count_above_P95','count_below_P5','max_value_in_segment','min_value_in_segment','max_breach_ratio'
    ])
    top20_segments = metric_fault_segments.head(20)
else:
    # sort
    anomaly_df = anomaly_df.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # compute breach ratio per point
    # high breach ratio = (value - p95) / (global_max - p95)
    # low breach ratio = (p5 - value) / (p5 - global_min)
    # guard denominators <= 0 by setting ratio 0 in that case
    denom_high = (anomaly_df['max_global'] - anomaly_df['p95']).astype(float)
    denom_low = (anomaly_df['p5'] - anomaly_df['min_global']).astype(float)

    high_ratio = np.where(denom_high > 0, (anomaly_df['value'] - anomaly_df['p95']) / denom_high, 0.0)
    low_ratio = np.where(denom_low > 0, (anomaly_df['p5'] - anomaly_df['value']) / denom_low, 0.0)

    # only positive breaches relevant; clip negatives to 0
    high_ratio = np.clip(high_ratio, 0.0, None)
    low_ratio = np.clip(low_ratio, 0.0, None)

    anomaly_df['breach_ratio'] = np.maximum(high_ratio, low_ratio)

    # identify consecutive segments per (cmdb_id,kpi_name) where timestamps differ exactly by 60s
    # For each group, compute diff and start new segment when diff != 60
    def assign_segments(g):
        g = g.sort_values('timestamp').copy()
        g['ts_diff'] = g['timestamp'].diff().fillna(9999999).astype(int)
        # start new segment when ts_diff != 60
        g['segment_id'] = (g['ts_diff'] != 60).cumsum()
        return g

    seg_df = anomaly_df.groupby(['cmdb_id','kpi_name'], sort=False).apply(assign_segments).reset_index(drop=True)

    # Aggregate per segment
    agg = seg_df.groupby(['cmdb_id','kpi_name','segment_id'], sort=False).agg(
        fault_start_epoch = ('timestamp','min'),
        fault_end_epoch = ('timestamp','max'),
        length_in_points = ('timestamp','count'),
        count_above_P95 = ('gt_p95','sum'),
        count_below_P5 = ('lt_p5','sum'),
        max_value_in_segment = ('value','max'),
        min_value_in_segment = ('value','min'),
        max_breach_ratio = ('breach_ratio','max')
    ).reset_index()

    # Convert epoch to local ISO strings (UTC+8)
    def to_local_iso(epoch):
        try:
            return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).isoformat()
        except Exception:
            return pd.NA

    agg['fault_start_local'] = agg['fault_start_epoch'].apply(to_local_iso)
    agg['fault_end_local'] = agg['fault_end_epoch'].apply(to_local_iso)

    # Rename component/kpi columns for output
    agg = agg.rename(columns={'cmdb_id':'component','kpi_name':'kpi_name'})

    # Filter segments: keep only those with length >=2 OR max_breach_ratio > 0.5
    filtered = agg[(agg['length_in_points'] >= 2) | (agg['max_breach_ratio'] > 0.5)].copy()

    # Select required fields and order
    metric_fault_segments = filtered[[
        'component','kpi_name','fault_start_local','fault_end_local','length_in_points',
        'count_above_P95','count_below_P5','max_value_in_segment','min_value_in_segment','max_breach_ratio'
    ]].copy()

    # Rank: first by length desc, then max_breach_ratio desc
    metric_fault_segments = metric_fault_segments.sort_values(['length_in_points','max_breach_ratio'], ascending=[False, False]).reset_index(drop=True)

    # Prepare compact top20 view
    top20_segments = metric_fault_segments.head(20)

# Store full segments in kernel variable as required
# metric_fault_segments already set
metric_fault_segments, top20_segments

```
Out[3]:
```
Summary of results

What I scanned
- I used the metric rows restricted to the allowed components and the previously computed global P95/P5 thresholds.
- I detected consecutive anomalous segments inside the incident window where consecutive means timestamps exactly 60s apart.
- An anomalous point is value > global P95 (high) or value < global P5 (low).
- I kept only segments with length >= 2 OR max_breach_ratio > 0.5.

Overall counts and storage
- Total anomalous segments found after filtering: 1,156 segments (stored in kernel variable metric_fault_segments).
- I return a compact top list (ranked by length desc, then max_breach_ratio desc). Up to 20 top segments were produced.

Top segments (examples from the top-ranked results)
- IG01 — JVM-Runtime_7778_JVM_JVM_Uptime
  - Segment 1: 2021-03-10T08:30:00+08:00 — 2021-03-10T08:31:00+08:00
    - length_in_points: 2, count_above_P95: 0, count_below_P5: 2
    - max_value: 415304.0, min_value: 415245.0, max_breach_ratio: 1.0
  - Segment 2: 2021-03-10T08:59:00+08:00 — 2021-03-10T09:00:00+08:00
    - length_in_points: 2, count_above_P95: 2, count_below_P5: 0
    - max_value: 417044.0, min_value: 416984.0, max_breach_ratio: 1.0

- IG01 — OSLinux-CPU_CPU_CPULoad
  - 2021-03-10T08:38:00+08:00 — 2021-03-10T08:39:00+08:00
    - length_in_points: 2, two low breaches, max_breach_ratio: 1.0

- IG01 — OSLinux-CPU_CPU_CPUUserTime
  - 2021-03-10T08:32:00+08:00 — 2021-03-10T08:33:00+08:00
    - length_in_points: 2, two low breaches, max_breach_ratio: 1.0

- IG01 — OSLinux-CPU_CPU_CPUWio
  - 2021-03-10T08:53:00+08:00 — 2021-03-10T08:54:00+08:00
    - length_in_points: 2, mixed high/low, max_breach_ratio: 1.0

- Many other top segments (MG01, MG02, IG02, Mysql01, Redis, apache02, etc.) appear with length 2 and max_breach_ratio = 1.0; several single-point segments passed the >0.5 breach ratio threshold and are included as well.

Fields returned for each segment
- component (cmdb_id)
- kpi_name
- fault_start_local, fault_end_local (local UTC+8 timestamps)
- length_in_points
- count_above_P95, count_below_P5
- max_value_in_segment, min_value_in_segment
- max_breach_ratio (maximum normalized breach within the segment)

Key takeaway
- There are many short consecutive anomalous segments (mostly length 2) across multiple components and KPIs. Top segments show clear, strong breaches (max_breach_ratio = 1.0), often with 2 points above P95 or 2 points below P5 within the 08:30–09:00 window.
- The complete segments table is available in the kernel variable metric_fault_segments for detailed inspection or export. If you want, I can display the top N segments or filter by component/KPI.

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                   kpi_name          fault_start_local            fault_end_local  length_in_points  count_above_P95  count_below_P5  max_value_in_segment  min_value_in_segment  max_breach_ratio
0         IG01            JVM-Runtime_7778_JVM_JVM_Uptime  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                0               2          4.153040e+05          4.152450e+05          1.000000
1         IG01            JVM-Runtime_7778_JVM_JVM_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          4.170440e+05          4.169840e+05          1.000000
2         IG01                    OSLinux-CPU_CPU_CPULoad  2021-03-10T08:38:00+08:00  2021-03-10T08:39:00+08:00                 2                0               2          1.000000e+00          1.000000e+00          1.000000
3         IG01                OSLinux-CPU_CPU_CPUUserTime  2021-03-10T08:32:00+08:00  2021-03-10T08:33:00+08:00                 2                0               2          2.517150e+01          2.516942e+01          1.000000
4         IG01                     OSLinux-CPU_CPU_CPUWio  2021-03-10T08:53:00+08:00  2021-03-10T08:54:00+08:00                 2                1               1          4.865000e-01          1.250000e-02          1.000000
...        ...                                        ...                        ...                        ...               ...              ...             ...                   ...                   ...               ...
1151   Redis01  redis-Redis_6379_Redis  (used_memory_rss)  2021-03-10T08:33:00+08:00  2021-03-10T08:33:00+08:00                 1                0               1          1.196442e+07          1.196442e+07          0.529412
1152   Redis01                OSLinux-CPU_CPU_CPUUserTime  2021-03-10T08:56:00+08:00  2021-03-10T08:56:00+08:00                 1                0               1          2.506659e+01          2.506659e+01          0.521119
1153   Redis01                 OSLinux-CPU_CPU_CPUSysTime  2021-03-10T08:58:00+08:00  2021-03-10T08:58:00+08:00                 1                1               0          1.675400e+00          1.675400e+00          0.518123
1154   Redis02                OSLinux-CPU_CPU_CPUUserTime  2021-03-10T08:32:00+08:00  2021-03-10T08:32:00+08:00                 1                0               1          1.748980e-01          1.748980e-01          0.506206
1155  apache02                 OSLinux-CPU_CPU_CPUSysTime  2021-03-10T08:31:00+08:00  2021-03-10T08:31:00+08:00                 1                0               1          1.911000e-01          1.911000e-01          0.503801

[1156 rows x 10 columns],    component                                      kpi_name          fault_start_local            fault_end_local  length_in_points  count_above_P95  count_below_P5  max_value_in_segment  min_value_in_segment  max_breach_ratio
0       IG01               JVM-Runtime_7778_JVM_JVM_Uptime  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                0               2          4.153040e+05          4.152450e+05               1.0
1       IG01               JVM-Runtime_7778_JVM_JVM_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          4.170440e+05          4.169840e+05               1.0
2       IG01                       OSLinux-CPU_CPU_CPULoad  2021-03-10T08:38:00+08:00  2021-03-10T08:39:00+08:00                 2                0               2          1.000000e+00          1.000000e+00               1.0
3       IG01                   OSLinux-CPU_CPU_CPUUserTime  2021-03-10T08:32:00+08:00  2021-03-10T08:33:00+08:00                 2                0               2          2.517150e+01          2.516942e+01               1.0
4       IG01                        OSLinux-CPU_CPU_CPUWio  2021-03-10T08:53:00+08:00  2021-03-10T08:54:00+08:00                 2                1               1          4.865000e-01          1.250000e-02               1.0
5       IG01            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                0               2          6.535260e+06          6.535200e+06               1.0
6       IG01            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          6.537007e+06          6.536946e+06               1.0
7       IG02                       OSLinux-CPU_CPU_CPULoad  2021-03-10T08:57:00+08:00  2021-03-10T08:58:00+08:00                 2                2               0          1.230000e+00          1.200000e+00               1.0
8       IG02                        OSLinux-CPU_CPU_CPUWio  2021-03-10T08:55:00+08:00  2021-03-10T08:56:00+08:00                 2                1               1          8.106760e-01          1.662800e-02               1.0
9       IG02            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                0               2          6.535315e+06          6.535254e+06               1.0
10      IG02            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          6.537060e+06          6.536999e+06               1.0
11      MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  2021-03-10T08:34:00+08:00  2021-03-10T08:35:00+08:00                 2                2               0          1.060421e+09          1.023870e+09               1.0
12      MG01               JVM-Runtime_7779_JVM_JVM_Uptime  2021-03-10T08:31:00+08:00  2021-03-10T08:32:00+08:00                 2                0               2          2.399378e+06          2.399318e+06               1.0
13      MG01               JVM-Runtime_7779_JVM_JVM_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          2.401058e+06          2.400998e+06               1.0
14      MG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                2               0          4.745150e+01          4.745150e+01               1.0
15      MG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                0               2          4.727180e+01          4.722040e+01               1.0
16      MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  2021-03-10T08:42:00+08:00  2021-03-10T08:43:00+08:00                 2                0               2          3.975935e+08          3.973913e+08               1.0
17   Mysql01                   OSLinux-CPU_CPU_CPUUserTime  2021-03-10T08:58:00+08:00  2021-03-10T08:59:00+08:00                 2                2               0          4.239000e-01          3.990020e-01               1.0
18   Mysql01            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:30:00+08:00  2021-03-10T08:31:00+08:00                 2                0               2          6.534867e+06          6.534806e+06               1.0
19   Mysql01            OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-10T08:59:00+08:00  2021-03-10T09:00:00+08:00                 2                2               0          6.536613e+06          6.536552e+06               1.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse metric_fault_segments from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_fault_segments exists
if 'metric_fault_segments' not in globals():
    metric_fault_segments = pd.DataFrame(columns=[
        'component','kpi_name','fault_start_local','fault_end_local','length_in_points',
        'count_above_P95','count_below_P5','max_value_in_segment','min_value_in_segment','max_breach_ratio'
    ])

# Parse fault_start_local into datetime for accurate earliest computation
# Some entries may already be timezone-aware ISO strings; pd.to_datetime will handle them.
mfs = metric_fault_segments.copy()
if not mfs.empty:
    mfs['fault_start_dt'] = pd.to_datetime(mfs['fault_start_local'], utc=True).dt.tz_convert(tz)
else:
    mfs['fault_start_dt'] = pd.NaT

# Aggregate by component
component_agg = mfs.groupby('component', sort=True).agg(
    total_segments = ('component','size'),
    max_length_in_points = ('length_in_points','max'),
    max_max_breach_ratio = ('max_breach_ratio','max'),
    earliest_fault_start_dt = ('fault_start_dt','min')
).reset_index()

# Convert earliest dt back to localized ISO string (or NA)
component_agg['earliest_fault_start_local'] = component_agg['earliest_fault_start_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else pd.NA
)

# Build KPI lists (up to 10 per component)
kpi_lists = (mfs.groupby('component')['kpi_name']
               .apply(lambda s: list(pd.Index(s.dropna().unique())[:10]))
               .reset_index()
               .rename(columns={'kpi_name':'kpi_list'}))

# Merge KPI lists into aggregation
component_fault_summary_full = component_agg.merge(kpi_lists, on='component', how='left')

# Select and rename columns for compact display
component_fault_summary_full = component_fault_summary_full[[
    'component',
    'total_segments',
    'max_length_in_points',
    'max_max_breach_ratio',
    'earliest_fault_start_local',
    'kpi_list'
]]

# Rank and take top 10 by total_segments desc then by max_max_breach_ratio desc
top10_components = component_fault_summary_full.sort_values(
    ['total_segments','max_max_breach_ratio'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# Keep output compact: show only requested fields (component + 4 aggregates) plus KPI list
top10_summary = top10_components.rename(columns={
    'max_max_breach_ratio': 'max_breach_ratio',
    'max_length_in_points': 'max_length_in_points',
    'total_segments': 'total_segments',
    'earliest_fault_start_local': 'earliest_fault_start_local',
    'kpi_list': 'kpi_list'
})[[
    'component',
    'total_segments',
    'max_length_in_points',
    'max_breach_ratio',
    'earliest_fault_start_local',
    'kpi_list'
]]

# Display the compact top-10 summary (and keep full aggregated table in component_fault_summary_full)
top10_summary, component_fault_summary_full.shape

```
Out[4]:
```
Summary (top 10 components by number of fault segments)

All times shown are local (UTC+8). For each component I show: total_segments, max_length_in_points, max_breach_ratio, earliest_fault_start_local, and example KPIs (up to 10) that contributed faults.

1. Mysql02
   - total_segments: 165
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: OSLinux-CPU_CPU_CPUSysTime, OSLinux-OSLinux_MEMORY_..., (up to 10 shown)

2. Mysql01
   - total_segments: 153
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: OSLinux-CPU_CPU_CPUUserTime, OSLinux-OSLinux_..., (up to 10 shown)

3. MG02
   - total_segments: 79
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: JVM-Memory_7779_JVM_Memory_HeapMemoryUsed, JVM-Runtime_..., ...

4. IG01
   - total_segments: 78
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: JVM-Runtime_7778_JVM_JVM_Uptime, OSLinux-CPU_CPU_..., ...

5. MG01
   - total_segments: 76
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: JVM-Memory_7779_JVM_Memory_HeapMemoryUsed, JVM-Runtime_..., ...

6. Tomcat01
   - total_segments: 76
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: Tomcat-Requests_7441-"http-nio-8003"_Processing..., ...

7. Tomcat03
   - total_segments: 74
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: Tomcat-Requests_7441-"http-nio-8003"_Processing..., ...

8. IG02
   - total_segments: 72
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: OSLinux-CPU_CPU_CPULoad, OSLinux-CPU_CPU_CPUWio, ...

9. Tomcat02
   - total_segments: 72
   - max_length_in_points: 2
   - max_breach_ratio: 1.0
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - example KPIs: Tomcat-Requests_7441-"http-nio-8003"_Processing..., ...

10. Redis01
    - total_segments: 69
    - max_length_in_points: 2
    - max_breach_ratio: 1.0
    - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
    - example KPIs: OSLinux-CPU_CPU_CPULoad, OSLinux-CPU_CPU_CPUUserTime, redis-Redis_6379_..., ...

Notes
- All top-10 components show mostly short segments (max length = 2) but with strong breaches (max_breach_ratio = 1.0).
- Earliest fault start for these components is 2021-03-10T08:30:00+08:00.
- KPI lists above show examples (each component's KPI list includes up to 10 distinct KPI names that produced segments).
- The full per-segment table is stored in kernel variable metric_fault_segments; the full per-component aggregation (all components) is also available if you want to see components beyond the top 10 or expand KPI lists.

The original code execution output of IPython Kernel is also provided below for reference:

(  component  total_segments  max_length_in_points  max_breach_ratio earliest_fault_start_local                                           kpi_list
0   Mysql02             165                     2               1.0  2021-03-10T08:30:00+08:00  [OSLinux-CPU_CPU_CPUSysTime, OSLinux-OSLinux_M...
1   Mysql01             153                     2               1.0  2021-03-10T08:30:00+08:00  [OSLinux-CPU_CPU_CPUUserTime, OSLinux-OSLinux_...
2      MG02              79                     2               1.0  2021-03-10T08:30:00+08:00  [JVM-Memory_7779_JVM_Memory_HeapMemoryUsed, JV...
3      IG01              78                     2               1.0  2021-03-10T08:30:00+08:00  [JVM-Runtime_7778_JVM_JVM_Uptime, OSLinux-CPU_...
4      MG01              76                     2               1.0  2021-03-10T08:30:00+08:00  [JVM-Memory_7779_JVM_Memory_HeapMemoryUsed, JV...
5  Tomcat01              76                     2               1.0  2021-03-10T08:30:00+08:00  [Tomcat-Requests_7441-"http-nio-8003"_Processi...
6  Tomcat03              74                     2               1.0  2021-03-10T08:30:00+08:00  [Tomcat-Requests_7441-"http-nio-8003"_Processi...
7      IG02              72                     2               1.0  2021-03-10T08:30:00+08:00  [OSLinux-CPU_CPU_CPULoad, OSLinux-CPU_CPU_CPUW...
8  Tomcat02              72                     2               1.0  2021-03-10T08:30:00+08:00  [Tomcat-Requests_7441-"http-nio-8003"_Processi...
9   Redis01              69                     2               1.0  2021-03-10T08:30:00+08:00  [OSLinux-CPU_CPU_CPULoad, OSLinux-CPU_CPU_CPUU..., (14, 6))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing variables: trace, tz, start_epoch, end_epoch
tz = pytz.timezone('Asia/Shanghai')

# Ensure start/end epoch exist
if 'start_epoch' not in globals() or 'end_epoch' not in globals():
    start_local = tz.localize(datetime(2021,3,10,8,30,0))
    end_local = tz.localize(datetime(2021,3,10,9,0,0))
    start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
    end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

top_components = ['Mysql02','Mysql01','MG02','IG01','MG01','Tomcat01','Tomcat03','IG02','Tomcat02','Redis01']

# Filter trace to allowed components and ensure numeric
trace_allowed = trace[trace['cmdb_id'].isin(top_components)].copy()
trace_allowed['value'] = pd.to_numeric(trace_allowed['value'], errors='coerce')

# Global thresholds per (cmdb_id, trace_name)
global_trace_thresh = trace_allowed.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_global='count',
    p95=lambda s: s.quantile(0.95),
    p5=lambda s: s.quantile(0.05)
).reset_index()

# Windowed rows
trace_window = trace_allowed[(trace_allowed['timestamp'] >= start_epoch) & (trace_allowed['timestamp'] <= end_epoch)].copy()

if trace_window.empty:
    # empty result
    trace_anomalies = pd.DataFrame(columns=[
        'cmdb_id','trace_name','points_inside','count_gt_p95','earliest_gt_local',
        'count_lt_p5','earliest_lt_local','max_inside','min_inside','total_anomalies'
    ])
    trace_top20 = trace_anomalies.head(20)
else:
    # merge thresholds
    trace_window = trace_window.merge(global_trace_thresh, on=['cmdb_id','trace_name'], how='left')
    trace_window['gt_p95'] = trace_window['value'] > trace_window['p95']
    trace_window['lt_p5'] = trace_window['value'] < trace_window['p5']

    # basic aggregates
    agg_basic = trace_window.groupby(['cmdb_id','trace_name'], sort=False).agg(
        points_inside=('value','count'),
        count_gt_p95=('gt_p95','sum'),
        count_lt_p5=('lt_p5','sum'),
        max_inside=('value','max'),
        min_inside=('value','min')
    ).reset_index()

    # earliest gt and lt epochs
    earliest_gt = (trace_window[trace_window['gt_p95']]
                   .groupby(['cmdb_id','trace_name'])['timestamp']
                   .min()
                   .reset_index()
                   .rename(columns={'timestamp':'earliest_gt_epoch'}))

    earliest_lt = (trace_window[trace_window['lt_p5']]
                   .groupby(['cmdb_id','trace_name'])['timestamp']
                   .min()
                   .reset_index()
                   .rename(columns={'timestamp':'earliest_lt_epoch'}))

    # merge earliest into agg_basic
    agg = agg_basic.merge(earliest_gt, on=['cmdb_id','trace_name'], how='left').merge(earliest_lt, on=['cmdb_id','trace_name'], how='left')

    # convert earliest epochs to local ISO strings
    def epoch_to_local_iso(val):
        if pd.isna(val):
            return pd.NA
        return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

    agg['earliest_gt_local'] = agg['earliest_gt_epoch'].apply(epoch_to_local_iso)
    agg['earliest_lt_local'] = agg['earliest_lt_epoch'].apply(epoch_to_local_iso)

    # total anomalies and earliest anomaly epoch for sorting
    agg['total_anomalies'] = agg['count_gt_p95'].astype(int) + agg['count_lt_p5'].astype(int)
    def min_epoch_or_large(row):
        e1 = row['earliest_gt_epoch']
        e2 = row['earliest_lt_epoch']
        if pd.isna(e1) and pd.isna(e2):
            return np.iinfo(np.int64).max
        vals = [v for v in (e1,e2) if pd.notna(v)]
        return int(min(vals)) if vals else np.iinfo(np.int64).max
    agg['earliest_anom_epoch_for_sort'] = agg.apply(min_epoch_or_large, axis=1)

    # sort and select display columns
    agg_sorted = agg.sort_values(['total_anomalies','earliest_anom_epoch_for_sort'], ascending=[False, True]).reset_index(drop=True)
    display_cols = ['cmdb_id','trace_name','points_inside','count_gt_p95','earliest_gt_local','count_lt_p5','earliest_lt_local','max_inside','min_inside']
    trace_top20 = agg_sorted[display_cols].head(20)

    # store full anomalies table
    trace_anomalies = agg_sorted.copy()

# Output compact top-20 and size of full anomalies table
trace_top20, trace_anomalies.shape

```
Out[5]:
```
Summary (trace analysis for selected components; incident window 2021-03-10 08:30–09:00 UTC+8)

What I did
- Restricted trace.csv to the listed top components and computed global P95/P5 per (cmdb_id, trace_name) using the full available series.
- Filtered trace rows to the incident window and aggregated per (cmdb_id, trace_name): points inside window, counts and earliest timestamps of >P95 and <P5, and max/min inside window.
- Ranked pairs by total anomalies (count_gt_p95 + count_lt_p5) and returned the top results. The full aggregated anomaly table is stored in kernel variable trace_anomalies.

Key results (top rows, up to 20 shown)
- Most top pairs have 30 points in the window (full coverage) and show 2 points above global P95 and 2 points below global P5 (total_anomalies = 4).
- Examples:
  - IG01 — trace.self.row_count: points_inside 30; 2 >P95 (earliest 2021-03-10T08:58:00+08:00); 2 <P5 (earliest 2021-03-10T08:30:00+08:00); max 1410; min 654.
  - IG01 — trace.to_Tomcat03.row_count: points_inside 30; 2 >P95 (08:58); 2 <P5 (08:30); max 167; min 72.
  - IG02 — trace.to_Tomcat03.duration_p95: points_inside 30; 2 >P95 (08:30); 2 <P5 (08:38); max 2.57995; min 1.025.
  - MG01 — trace.from_Tomcat04.duration_p95: points_inside 30; 2 >P95 (08:33); 2 <P5 (08:30); max 2.358; min 0.1094.
  - Tomcat03 — trace.self.row_count: points_inside 30; 2 >P95 (08:58); 2 <P5 (08:30); max 2368; min 961.

Counts and storage
- The aggregated anomaly table contains 174 (cmdb_id, trace_name) groups (trace_anomalies shape = (174, 13)).
- Top 20 displayed groups are included in the execution output above; each row shows the requested aggregates (points_inside, counts and earliest anomaly times, max/min inside).

Takeaway
- Multiple trace features across IG01, IG02, MG01, MG02, Tomcat03, etc., show both high and low breaches within the incident window (commonly 2 high and 2 low points per trace).
- The full per-pair anomaly table is stored in trace_anomalies for further inspection or export.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  points_inside  count_gt_p95          earliest_gt_local  count_lt_p5          earliest_lt_local   max_inside  min_inside
0       IG01               trace.self.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00  1410.000000  654.000000
1       IG01        trace.to_Tomcat03.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00   167.000000   72.000000
2       IG02     trace.to_Tomcat03.duration_p95             30             2  2021-03-10T08:30:00+08:00            2  2021-03-10T08:38:00+08:00     2.579950    1.025000
3       IG02     trace.to_Tomcat04.duration_p95             30             2  2021-03-10T08:54:00+08:00            2  2021-03-10T08:30:00+08:00     2.156000    0.967000
4       MG01      trace.from_Tomcat03.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00   189.000000   74.000000
5       MG01   trace.from_Tomcat04.duration_p95             30             2  2021-03-10T08:33:00+08:00            2  2021-03-10T08:30:00+08:00     2.358000    0.109400
6       MG02  trace.from_dockerB2.duration_mean             30             2  2021-03-10T08:30:00+08:00            2  2021-03-10T08:39:00+08:00     0.076678    0.005947
7       MG02    trace.to_dockerA1.duration_mean             30             2  2021-03-10T08:43:00+08:00            2  2021-03-10T08:30:00+08:00     0.332413    0.065542
8       MG02     trace.to_dockerA1.duration_p95             30             2  2021-03-10T08:43:00+08:00            2  2021-03-10T08:30:00+08:00     1.125700    0.302200
9   Tomcat03          trace.from_IG01.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00   167.000000   72.000000
10  Tomcat03       trace.from_IG02.duration_p95             30             2  2021-03-10T08:30:00+08:00            2  2021-03-10T08:38:00+08:00     2.579950    1.025000
11  Tomcat03               trace.self.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00  2368.000000  961.000000
12  Tomcat03            trace.to_MG01.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:30:00+08:00   189.000000   74.000000
13      IG01        trace.to_Tomcat04.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:31:00+08:00   191.000000   75.000000
14      IG02        trace.to_Tomcat01.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:31:00+08:00   186.000000   83.000000
15      MG01        trace.to_dockerB2.row_count             30             2  2021-03-10T08:43:00+08:00            2  2021-03-10T08:31:00+08:00   151.000000   51.000000
16      MG02      trace.from_Tomcat01.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:31:00+08:00   183.000000   74.000000
17      MG02      trace.from_Tomcat04.row_count             30             2  2021-03-10T08:58:00+08:00            2  2021-03-10T08:31:00+08:00   207.000000   77.000000
18      MG02    trace.to_dockerB2.duration_mean             30             2  2021-03-10T08:31:00+08:00            2  2021-03-10T08:32:00+08:00     0.497565    0.040777
19      MG02     trace.to_dockerB2.duration_p95             30             2  2021-03-10T08:31:00+08:00            2  2021-03-10T08:32:00+08:00     2.177000    0.094800, (174, 13))```
```

In [ ]:
# In[6]:

import pandas as pd
import re
import pytz
import numpy as np

# reuse variables from kernel: trace_anomalies, metric_fault_segments, allowed, tz
# ensure they exist
if 'trace_anomalies' not in globals():
    trace_anomalies = pd.DataFrame()
if 'metric_fault_segments' not in globals():
    metric_fault_segments = pd.DataFrame()
if 'allowed' not in globals():
    allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
               'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
tz = pytz.timezone('Asia/Shanghai')

ta = trace_anomalies.copy()

# If ta is empty, return empty results
if ta.empty:
    trace_referenced_component_summary_full = pd.DataFrame(columns=[
        'component_name','anomalous_trace_group_count','sum_anomalies','earliest_anomaly_local','has_metric_faults'
    ])
    top10_ref_summary = trace_referenced_component_summary_full.head(10)
else:
    # extract referenced component from trace_name using regex for to_<COMP> or from_<COMP>
    def extract_ref_component(name):
        if pd.isna(name):
            return pd.NA
        m = re.search(r'\b(?:to|from)_([A-Za-z0-9_]+)', name)
        return m.group(1) if m else pd.NA

    ta['referenced_component'] = ta['trace_name'].apply(extract_ref_component)

    # keep only rows where referenced_component is in allowed candidate list
    ta_ref = ta[ta['referenced_component'].isin(allowed)].copy()

    # compute total anomalies per row (ensure numeric)
    ta_ref['count_gt_p95'] = pd.to_numeric(ta_ref.get('count_gt_p95', 0), errors='coerce').fillna(0).astype(int)
    ta_ref['count_lt_p5'] = pd.to_numeric(ta_ref.get('count_lt_p5', 0), errors='coerce').fillna(0).astype(int)
    ta_ref['total_anomalies_row'] = ta_ref['count_gt_p95'] + ta_ref['count_lt_p5']

    # compute earliest anomaly epoch per row (min of earliest_gt_epoch and earliest_lt_epoch)
    # use epoch columns if present; otherwise try parsing earliest_gt_local/earliest_lt_local
    def row_earliest_epoch(row):
        e1 = row.get('earliest_gt_epoch', np.nan)
        e2 = row.get('earliest_lt_epoch', np.nan)
        # handle NaN/NA
        vals = []
        for v in (e1, e2):
            if pd.notna(v):
                try:
                    vals.append(int(v))
                except Exception:
                    pass
        if vals:
            return min(vals)
        # fallback: try parsing local strings
        for key in ('earliest_gt_local','earliest_lt_local'):
            v = row.get(key, None)
            if pd.notna(v) and v is not None:
                try:
                    ep = int(pd.to_datetime(v, utc=True).tz_convert(tz).tz_convert(pytz.UTC).timestamp())
                    return ep
                except Exception:
                    pass
        return np.nan

    ta_ref['earliest_anom_epoch_row'] = ta_ref.apply(row_earliest_epoch, axis=1)

    # Aggregate by referenced_component
    agg = ta_ref.groupby('referenced_component', sort=False).agg(
        anomalous_trace_group_count = ('trace_name','nunique'),  # number of distinct trace groups referencing it
        sum_anomalies = ('total_anomalies_row','sum'),
        earliest_anom_epoch = ('earliest_anom_epoch_row','min')
    ).reset_index().rename(columns={'referenced_component':'component_name'})

    # convert earliest epoch to local ISO
    def epoch_to_local_iso(e):
        if pd.isna(e):
            return pd.NA
        try:
            return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat()
        except Exception:
            return pd.NA

    agg['earliest_anomaly_local'] = agg['earliest_anom_epoch'].apply(epoch_to_local_iso)

    # verify presence in metric_fault_segments
    metric_components = set(metric_fault_segments['component'].dropna().unique()) if not metric_fault_segments.empty else set()
    agg['has_metric_faults'] = agg['component_name'].apply(lambda c: c in metric_components)

    # final selection and sorting: by anomalous_trace_group_count desc, then sum_anomalies desc
    trace_referenced_component_summary_full = agg[['component_name','anomalous_trace_group_count','sum_anomalies','earliest_anomaly_local','has_metric_faults']].copy()
    top10_ref_summary = trace_referenced_component_summary_full.sort_values(
        ['anomalous_trace_group_count','sum_anomalies'], ascending=[False, False]
    ).head(10).reset_index(drop=True)

# Display compact top-10 summary and keep full aggregated table in trace_referenced_component_summary_full
top10_ref_summary, trace_referenced_component_summary_full.shape

```
Out[6]:
```
Summary of referenced-component aggregation (extracted "to_X" / "from_X" from trace names; limited to candidate components):

Method: for each referenced component X I counted distinct anomalous trace groups that reference X, summed their anomaly counts (gt_P95 + lt_P5), found the earliest anomaly time (local UTC+8), and checked whether X has metric fault segments.

Top referenced components (ranked by anomalous_trace_group_count desc, then sum_anomalies desc):

1. Tomcat04 — anomalous_trace_group_count: 6, sum_anomalies: 48, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
2. Tomcat01 — anomalous_trace_group_count: 6, sum_anomalies: 48, earliest_anomaly_local: 2021-03-10T08:31:00+08:00, has_metric_faults: True  
3. Tomcat02 — anomalous_trace_group_count: 6, sum_anomalies: 46, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
4. Tomcat03 — anomalous_trace_group_count: 6, sum_anomalies: 45, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
5. MG01     — anomalous_trace_group_count: 3, sum_anomalies: 36, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
6. IG01     — anomalous_trace_group_count: 3, sum_anomalies: 35, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
7. IG02     — anomalous_trace_group_count: 3, sum_anomalies: 34, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True  
8. MG02     — anomalous_trace_group_count: 3, sum_anomalies: 34, earliest_anomaly_local: 2021-03-10T08:30:00+08:00, has_metric_faults: True

Notes:
- Eight referenced components were found; the top four are Tomcat04/01/02/03 (each referenced by 6 anomalous trace groups).  
- All listed referenced components have corresponding metric fault segments (has_metric_faults = True).  
- Earliest anomalies for most components occurred at 2021-03-10T08:30:00+08:00 (Tomcat01 earliest at 08:31).

The original code execution output of IPython Kernel is also provided below for reference:

(  component_name  anomalous_trace_group_count  sum_anomalies     earliest_anomaly_local  has_metric_faults
0       Tomcat04                            6             48  2021-03-10T08:30:00+08:00               True
1       Tomcat01                            6             48  2021-03-10T08:31:00+08:00               True
2       Tomcat02                            6             46  2021-03-10T08:30:00+08:00               True
3       Tomcat03                            6             45  2021-03-10T08:30:00+08:00               True
4           MG01                            3             36  2021-03-10T08:30:00+08:00               True
5           IG01                            3             35  2021-03-10T08:30:00+08:00               True
6           IG02                            3             34  2021-03-10T08:30:00+08:00               True
7           MG02                            3             34  2021-03-10T08:30:00+08:00               True, (8, 5))```
```

In [ ]:
# In[7]:

import pandas as pd
import re
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
if 'trace_anomalies' not in globals():
    trace_anomalies = pd.DataFrame()
if 'metric_fault_segments' not in globals():
    metric_fault_segments = pd.DataFrame()

ta = trace_anomalies.copy()

# If empty, create empty results
if ta.empty:
    trace_edges = pd.DataFrame(columns=[
        'edge_source','edge_target','aggregated_trace_group_count',
        'aggregated_sum_anomalies','earliest_anomaly_epoch'
    ])
    top20_edges = trace_edges.head(20)
    tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']
    tomcat_incoming_summary = pd.DataFrame([{
        'edge_target': t,
        'incoming_edge_count': 0,
        'total_incoming_trace_groups': 0,
        'total_incoming_anomalies': 0,
        'earliest_incoming_anomaly_local': pd.NA
    } for t in tomcats])
else:
    # Helper to extract referenced component from trace_name
    def extract_ref(name):
        if pd.isna(name):
            return pd.NA, pd.NA
        m_to = re.search(r'\bto_([A-Za-z0-9_]+)', name)
        m_from = re.search(r'\bfrom_([A-Za-z0-9_]+)', name)
        if m_to:
            return 'to', m_to.group(1)
        if m_from:
            return 'from', m_from.group(1)
        return pd.NA, pd.NA

    # Apply extraction
    extracted = ta['trace_name'].apply(lambda x: pd.Series(extract_ref(x), index=['dir','ref_comp']))
    ta = pd.concat([ta, extracted], axis=1)

    # Keep only rows where a ref_comp was found
    ta = ta[ta['ref_comp'].notna()].copy()

    # Ensure anomaly counts numeric
    ta['count_gt_p95'] = pd.to_numeric(ta.get('count_gt_p95', 0), errors='coerce').fillna(0).astype(int)
    ta['count_lt_p5'] = pd.to_numeric(ta.get('count_lt_p5', 0), errors='coerce').fillna(0).astype(int)
    ta['total_anomalies_for_group'] = ta['count_gt_p95'] + ta['count_lt_p5']

    # Determine earliest anomaly epoch per row (prefer epoch fields if present)
    def get_earliest_epoch(row):
        # try epoch columns first
        e_candidates = []
        for col in ('earliest_gt_epoch','earliest_lt_epoch'):
            v = row.get(col, None)
            if pd.notna(v):
                try:
                    e_candidates.append(int(v))
                except Exception:
                    pass
        # fallback to local ISO strings
        for col in ('earliest_gt_local','earliest_lt_local'):
            v = row.get(col, None)
            if pd.notna(v):
                try:
                    # parse as UTC-aware then convert to UTC epoch
                    dt = pd.to_datetime(v, utc=True)
                    e_candidates.append(int(dt.tz_convert(pytz.UTC).timestamp()))
                except Exception:
                    pass
        if not e_candidates:
            return np.nan
        return int(min(e_candidates))

    ta['earliest_anom_epoch'] = ta.apply(get_earliest_epoch, axis=1)

    # Build directed edges per row
    def build_edge(row):
        if row['dir'] == 'to':
            return row['cmdb_id'], row['ref_comp']
        elif row['dir'] == 'from':
            return row['ref_comp'], row['cmdb_id']
        else:
            return pd.NA, pd.NA

    edges = ta.apply(lambda r: pd.Series(build_edge(r), index=['edge_source','edge_target']), axis=1)
    ta = pd.concat([ta, edges], axis=1)

    # Each trace_anomalies row counts as one anomalous trace group
    ta['anomalous_trace_group_count'] = 1

    # Prepare raw edges DataFrame
    edges_raw = ta[['edge_source','edge_target','anomalous_trace_group_count','total_anomalies_for_group','earliest_anom_epoch']].copy()

    # Drop rows with missing edge_source/edge_target
    edges_raw = edges_raw.dropna(subset=['edge_source','edge_target'])

    # Aggregate by (edge_source, edge_target)
    trace_edges = edges_raw.groupby(['edge_source','edge_target'], sort=False).agg(
        aggregated_trace_group_count = ('anomalous_trace_group_count','sum'),
        aggregated_sum_anomalies = ('total_anomalies_for_group','sum'),
        earliest_anomaly_epoch = ('earliest_anom_epoch','min')
    ).reset_index()

    # Convert earliest epoch to local ISO
    def epoch_to_local_iso(e):
        if pd.isna(e):
            return pd.NA
        try:
            return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat()
        except Exception:
            return pd.NA

    trace_edges['earliest_anomaly_local'] = trace_edges['earliest_anomaly_epoch'].apply(epoch_to_local_iso)

    # Keep only edges where edge_target is one of the Tomcats
    tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']
    edges_to_tomcats = trace_edges[trace_edges['edge_target'].isin(tomcats)].copy()

    # Prepare compact edge table for display (limit top 20)
    top20_edges = edges_to_tomcats.sort_values(
        ['aggregated_trace_group_count','aggregated_sum_anomalies'],
        ascending=[False, False]
    ).head(20)[['edge_source','edge_target','aggregated_trace_group_count','aggregated_sum_anomalies','earliest_anomaly_local']].reset_index(drop=True)

    # Per-Tomcat incoming summary
    incoming_rows = []
    for t in tomcats:
        subset = trace_edges[trace_edges['edge_target']==t]
        if subset.empty:
            incoming_rows.append({
                'edge_target': t,
                'incoming_edge_count': 0,
                'total_incoming_trace_groups': 0,
                'total_incoming_anomalies': 0,
                'earliest_incoming_anomaly_local': pd.NA
            })
        else:
            incoming_edge_count = subset['edge_source'].nunique()
            total_incoming_trace_groups = int(subset['aggregated_trace_group_count'].sum())
            total_incoming_anomalies = int(subset['aggregated_sum_anomalies'].sum())
            earliest_epoch = subset['earliest_anomaly_epoch'].min()
            earliest_local = epoch_to_local_iso(earliest_epoch)
            incoming_rows.append({
                'edge_target': t,
                'incoming_edge_count': int(incoming_edge_count),
                'total_incoming_trace_groups': total_incoming_trace_groups,
                'total_incoming_anomalies': total_incoming_anomalies,
                'earliest_incoming_anomaly_local': earliest_local
            })

    tomcat_incoming_summary = pd.DataFrame(incoming_rows)

# Final displayed compact outputs (store full aggregated edge table in trace_edges)
trace_edges, top20_edges, tomcat_incoming_summary

```
Out[7]:
```
Summary of directed anomalous trace edges targeting the Tomcats (edge = source -> target). Each edge shows: aggregated_trace_group_count, aggregated_sum_anomalies, earliest_anomaly_local.

Top edges with Tomcat targets (compact):
- IG01 -> Tomcat03: 6 trace groups, 24 total anomalies, earliest 2021-03-10T08:30:00+08:00
- IG02 -> Tomcat01: 6 trace groups, 24 total anomalies, earliest 2021-03-10T08:31:00+08:00
- IG02 -> Tomcat02: 6 trace groups, 24 total anomalies, earliest 2021-03-10T08:32:00+08:00
- IG01 -> Tomcat01: 6 trace groups, 24 total anomalies, earliest 2021-03-10T08:34:00+08:00
- IG01 -> Tomcat02: 6 trace groups, 22 total anomalies, earliest 2021-03-10T08:32:00+08:00
- IG02 -> Tomcat03: 6 trace groups, 20 total anomalies, earliest 2021-03-10T08:30:00+08:00
- IG02 -> Tomcat04: 3 trace groups, 12 total anomalies, earliest 2021-03-10T08:30:00+08:00
- IG01 -> Tomcat04: 3 trace groups, 12 total anomalies, earliest 2021-03-10T08:31:00+08:00
- root -> Tomcat02: 3 trace groups, 6 total anomalies, earliest 2021-03-10T08:30:00+08:00
- root -> Tomcat01: 3 trace groups, 0 total anomalies, earliest: (no timestamp available)

Per-Tomcat incoming summary (four Tomcats):
- Tomcat01: incoming_edge_count = 3 distinct sources, total_incoming_trace_groups = 15, total_incoming_anomalies = 48, earliest_incoming_anomaly_local = 2021-03-10T08:31:00+08:00
- Tomcat02: incoming_edge_count = 3, total_incoming_trace_groups = 15, total_incoming_anomalies = 52, earliest_incoming_anomaly_local = 2021-03-10T08:30:00+08:00
- Tomcat03: incoming_edge_count = 3, total_incoming_trace_groups = 15, total_incoming_anomalies = 44, earliest_incoming_anomaly_local = 2021-03-10T08:30:00+08:00
- Tomcat04: incoming_edge_count = 2, total_incoming_trace_groups = 6,  total_incoming_anomalies = 24, earliest_incoming_anomaly_local = 2021-03-10T08:30:00+08:00

Notes
- Earliest timestamps are local (UTC+8).
- The full aggregated edge table is stored in the kernel variable trace_edges if you want the complete list of edges and details.

The original code execution output of IPython Kernel is also provided below for reference:

(   edge_source edge_target  aggregated_trace_group_count  aggregated_sum_anomalies  earliest_anomaly_epoch     earliest_anomaly_local
0         IG01    Tomcat03                             6                        24            1.615336e+09  2021-03-10T08:30:00+08:00
1         IG02    Tomcat03                             6                        20            1.615336e+09  2021-03-10T08:30:00+08:00
2         IG02    Tomcat04                             3                        12            1.615336e+09  2021-03-10T08:30:00+08:00
3     Tomcat03        MG01                             6                        24            1.615336e+09  2021-03-10T08:30:00+08:00
4     Tomcat04        MG01                             3                        12            1.615336e+09  2021-03-10T08:30:00+08:00
..         ...         ...                           ...                       ...                     ...                        ...
34        root        IG02                             3                         0                     NaN                       <NA>
35        root        MG01                             3                         0                     NaN                       <NA>
36        root        MG02                             3                         0                     NaN                       <NA>
37        root    Tomcat01                             3                         0                     NaN                       <NA>
38        root    Tomcat03                             3                         0                     NaN                       <NA>

[39 rows x 6 columns],    edge_source edge_target  aggregated_trace_group_count  aggregated_sum_anomalies     earliest_anomaly_local
0         IG01    Tomcat03                             6                        24  2021-03-10T08:30:00+08:00
1         IG02    Tomcat01                             6                        24  2021-03-10T08:31:00+08:00
2         IG02    Tomcat02                             6                        24  2021-03-10T08:32:00+08:00
3         IG01    Tomcat01                             6                        24  2021-03-10T08:34:00+08:00
4         IG01    Tomcat02                             6                        22  2021-03-10T08:32:00+08:00
5         IG02    Tomcat03                             6                        20  2021-03-10T08:30:00+08:00
6         IG02    Tomcat04                             3                        12  2021-03-10T08:30:00+08:00
7         IG01    Tomcat04                             3                        12  2021-03-10T08:31:00+08:00
8         root    Tomcat02                             3                         6  2021-03-10T08:30:00+08:00
9         root    Tomcat01                             3                         0                       <NA>
10        root    Tomcat03                             3                         0                       <NA>,   edge_target  incoming_edge_count  total_incoming_trace_groups  total_incoming_anomalies earliest_incoming_anomaly_local
0    Tomcat01                    3                           15                        48       2021-03-10T08:31:00+08:00
1    Tomcat02                    3                           15                        52       2021-03-10T08:30:00+08:00
2    Tomcat03                    3                           15                        44       2021-03-10T08:30:00+08:00
3    Tomcat04                    2                            6                        24       2021-03-10T08:30:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing kernel variables if present: metric_fault_segments, log, error_logs, tz, start_epoch, end_epoch
tz = pytz.timezone('Asia/Shanghai')
if 'start_epoch' not in globals() or 'end_epoch' not in globals():
    start_local = tz.localize(datetime(2021,3,10,8,30,0))
    end_local = tz.localize(datetime(2021,3,10,9,0,0))
    start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
    end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Target components
components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','Mysql01','Mysql02']

# Ensure metric_fault_segments exists
if 'metric_fault_segments' not in globals():
    metric_fault_segments = pd.DataFrame(columns=[
        'component','kpi_name','fault_start_local','fault_end_local','length_in_points',
        'count_above_P95','count_below_P5','max_value_in_segment','min_value_in_segment','max_breach_ratio'
    ])

mfs = metric_fault_segments.copy()

# Part 1: Aggregate metric_fault_segments per component
mfs_target = mfs[mfs['component'].isin(components)].copy()

if not mfs_target.empty:
    mfs_target['fault_start_dt'] = pd.to_datetime(mfs_target['fault_start_local'], utc=True).dt.tz_convert(tz)
else:
    mfs_target['fault_start_dt'] = pd.NaT

component_agg = mfs_target.groupby('component', sort=False).agg(
    total_segments = ('component','size'),
    earliest_fault_start_dt = ('fault_start_dt','min'),
    max_breach_ratio = ('max_breach_ratio','max')
).reset_index()

component_agg['earliest_fault_start_local'] = component_agg['earliest_fault_start_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else pd.NA
)

# Top 5 KPIs per component with their segment counts
kpi_counts = mfs_target.groupby(['component','kpi_name']).size().reset_index(name='segment_count')

def top_kpis_for_component(df):
    df_sorted = df.sort_values('segment_count', ascending=False).head(5)
    return ", ".join([f"{row.kpi_name}:{int(row.segment_count)}" for row in df_sorted.itertuples(index=False)])

if not kpi_counts.empty:
    top_kpis_series = kpi_counts.groupby('component').apply(top_kpis_for_component).reset_index().rename(columns={0:'top_kpis'})
else:
    top_kpis_series = pd.DataFrame(columns=['component','top_kpis'])

component_metric_summary = component_agg.merge(top_kpis_series, on='component', how='left')

# Ensure all target components appear
all_comp_df = pd.DataFrame({'component':components})
component_metric_summary = all_comp_df.merge(component_metric_summary, on='component', how='left')
component_metric_summary['total_segments'] = component_metric_summary['total_segments'].fillna(0).astype(int)
component_metric_summary['max_breach_ratio'] = component_metric_summary['max_breach_ratio'].fillna(0.0)
component_metric_summary['earliest_fault_start_local'] = component_metric_summary['earliest_fault_start_local'].fillna(pd.NA)
component_metric_summary['top_kpis'] = component_metric_summary['top_kpis'].fillna("")

metric_table = component_metric_summary[[
    'component','total_segments','earliest_fault_start_local','max_breach_ratio','top_kpis'
]]

# Part 2: Logs analysis for same components and incident window
if 'log' not in globals():
    log = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
if 'error_logs' not in globals():
    error_logs = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

component_logs_filtered = log[
    (log['cmdb_id'].isin(components)) &
    (log['timestamp'] >= start_epoch) &
    (log['timestamp'] <= end_epoch)
].copy()

component_logs_filtered['value'] = pd.to_numeric(component_logs_filtered['value'], errors='coerce')

def compute_log_aggregates(df):
    rows = []
    for comp in components:
        sub = df[df['cmdb_id']==comp]
        r = {
            'component': comp,
            'log_rows': int(sub.shape[0])
        }
        err_sub = sub[sub['log_name']=='log.error_count']
        r['sum_error_count'] = float(err_sub['value'].sum()) if not err_sub.empty else 0.0
        err_pos = err_sub[err_sub['value']>0]
        if not err_pos.empty:
            ep = int(err_pos['timestamp'].min())
            r['earliest_error_count_local'] = pd.to_datetime(ep, unit='s', utc=True).tz_convert(tz).isoformat()
        else:
            r['earliest_error_count_local'] = pd.NA
        name_counts = sub.groupby('log_name').size().reset_index(name='count').sort_values('count', ascending=False).head(10)
        if not name_counts.empty:
            r['top_log_names'] = ", ".join([f"{t.log_name}:{int(t.count)}" for t in name_counts.itertuples(index=False)])
        else:
            r['top_log_names'] = ""
        rows.append(r)
    return pd.DataFrame(rows)

component_logs_summary = compute_log_aggregates(component_logs_filtered)

# error_logs counts per component
err = error_logs.copy()
err_counts = err[err['cmdb_id'].isin(components)].groupby('cmdb_id').size().reset_index(name='error_log_count')
err_all = pd.DataFrame({'component':components}).merge(err_counts, left_on='component', right_on='cmdb_id', how='left').drop(columns=['cmdb_id'])
err_all['error_log_count'] = err_all['error_log_count'].fillna(0).astype(int)

component_logs_summary = component_logs_summary.merge(err_all, on='component', how='left')

# Keep compact ordering
component_logs_summary = component_logs_summary[[
    'component','log_rows','sum_error_count','earliest_error_count_local','top_log_names','error_log_count'
]]

# Store the requested kernel variables
# component_metric_summary and component_logs_filtered are stored as variables already
# (component_metric_summary contains full per-component metric aggregation; component_logs_filtered stores filtered logs)
component_metric_summary_full = component_metric_summary
component_logs_filtered_full = component_logs_filtered

# Display compact results
metric_table, component_logs_summary.head(20)

```
Out[8]:
```
Summary (incident window 2021-03-10 08:30–09:00 local time UTC+8)

Part 1 — metric fault segments (aggregated per component)
For each of the eight components I report: total_segments, earliest_fault_start_local, max_breach_ratio, and top-5 KPI contributors (KPI:segment_count).

- Tomcat01
  - total_segments: 76
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU-... (top 5 shown in stored result)

- Tomcat02
  - total_segments: 72
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU-...

- Tomcat03
  - total_segments: 74
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU-...

- Tomcat04
  - total_segments: 69
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): OSLinux-CPU_CPU-1_SingleCpuidle:2, OSLinux-CPU-...

- MG01
  - total_segments: 76
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): JVM-Memory_7779_JVM_Memory_HeapMemoryUsage:2, JVM-Runtime_...

- MG02
  - total_segments: 79
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): JVM-Memory_7779_JVM_Memory_HeapMemoryUsage:2, JVM-Runtime_...

- Mysql01
  - total_segments: 153
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): Mysql-MySQL_3306_Binlog Cache Use:2, Mysql-MySQL_...

- Mysql02
  - total_segments: 165
  - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
  - max_breach_ratio: 1.0
  - top KPIs (examples): Mysql-MySQL_3306_Binlog Cache Use:2, Mysql-MySQL_...

(Full per-component metric aggregation saved in kernel variable component_metric_summary.)

Part 2 — logs (filtered to same 8 components and the incident window)
Per component I report: log_rows (rows in window), sum_value_of log.error_count, earliest local timestamp where log.error_count > 0 (if any), top log_name counts (up to top 10), and error_logs count for that component.

- Tomcat01
  - log_rows: 75
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top log_name counts (examples): log.error_count:31, log.row_count:31, log.gc_pause_max:...
  - error_logs count for Tomcat01: 0

- Tomcat02
  - log_rows: 83
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top log_name counts (examples): log.error_count:31, log.row_count:31, log.gc_pause_max:...
  - error_logs count for Tomcat02: 0

- Tomcat03
  - log_rows: 75
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top log_name counts (examples): log.error_count:31, log.row_count:31, ...
  - error_logs count for Tomcat03: 0

- Tomcat04
  - log_rows: 74
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top log_name counts (examples): log.error_count:31, log.row_count:31, ...
  - error_logs count for Tomcat04: 0

- MG01
  - log_rows: 0
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top_log_names: none
  - error_logs count for MG01: 0

- MG02
  - log_rows: 0
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top_log_names: none
  - error_logs count for MG02: 0

- Mysql01
  - log_rows: 38
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top log_name counts (examples): log.error_count:19, log.row_count:19
  - error_logs count for Mysql01: 0

- Mysql02
  - log_rows: 0
  - sum of log.error_count values: 0.0
  - earliest log.error_count > 0: none (NA)
  - top_log_names: none
  - error_logs count for Mysql02: 0

(Filtered logs for these components are saved in kernel variable component_logs_filtered.)

Key observations
- All eight components show metric fault segments in the incident window; Mysql01/Mysql02 have the largest counts (153/165 segments).
- max_breach_ratio = 1.0 for all eight components (strong breaches observed).
- Several Tomcats have many log rows (≈74–83) but the sum of log.error_count is zero in the window (log.error_count rows exist but their values are zero), and no earliest log.error_count>0 was found.
- error_logs.csv contains no error_log entries for these components (counts = 0).

If you want, I can (a) expand any component’s top KPI list (show the full top 5), (b) show the component_metric_summary or component_logs_filtered variables, or (c) extract the exact KPI names and counts per component in a more readable list.

The original code execution output of IPython Kernel is also provided below for reference:

(  component  total_segments earliest_fault_start_local  max_breach_ratio                                           top_kpis
0  Tomcat01              76  2021-03-10T08:30:00+08:00               1.0  OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU...
1  Tomcat02              72  2021-03-10T08:30:00+08:00               1.0  OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU...
2  Tomcat03              74  2021-03-10T08:30:00+08:00               1.0  OSLinux-CPU_CPU-0_SingleCpuUtil:2, OSLinux-CPU...
3  Tomcat04              69  2021-03-10T08:30:00+08:00               1.0  OSLinux-CPU_CPU-1_SingleCpuidle:2, OSLinux-CPU...
4      MG01              76  2021-03-10T08:30:00+08:00               1.0  JVM-Memory_7779_JVM_Memory_HeapMemoryUsage:2, ...
5      MG02              79  2021-03-10T08:30:00+08:00               1.0  JVM-Memory_7779_JVM_Memory_HeapMemoryUsage:2, ...
6   Mysql01             153  2021-03-10T08:30:00+08:00               1.0  Mysql-MySQL_3306_Binlog Cache Use:2, Mysql-MyS...
7   Mysql02             165  2021-03-10T08:30:00+08:00               1.0  Mysql-MySQL_3306_Binlog Cache Use:2, Mysql-MyS...,   component  log_rows  sum_error_count earliest_error_count_local                                      top_log_names  error_log_count
0  Tomcat01        75              0.0                       <NA>  log.error_count:31, log.row_count:31, log.gc_p...                0
1  Tomcat02        83              0.0                       <NA>  log.error_count:31, log.row_count:31, log.gc_p...                0
2  Tomcat03        75              0.0                       <NA>  log.error_count:31, log.row_count:31, log.gc_p...                0
3  Tomcat04        74              0.0                       <NA>  log.error_count:31, log.row_count:31, log.gc_p...                0
4      MG01         0              0.0                       <NA>                                                                   0
5      MG02         0              0.0                       <NA>                                                                   0
6   Mysql01        38              0.0                       <NA>               log.error_count:19, log.row_count:19                0
7   Mysql02         0              0.0                       <NA>                                                                   0)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from collections import Counter

# Reuse kernel variables: metric_fault_segments, trace_edges
tz = pytz.timezone('Asia/Shanghai')

components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','Mysql01','Mysql02']

# Ensure metric_fault_segments exists
if 'metric_fault_segments' not in globals():
    metric_fault_segments = pd.DataFrame(columns=[
        'component','kpi_name','fault_start_local','fault_end_local','length_in_points',
        'count_above_P95','count_below_P5','max_value_in_segment','min_value_in_segment','max_breach_ratio'
    ])

mfs = metric_fault_segments.copy()

# Normalize fault_start_local to datetime (UTC+8) for comparisons
if not mfs.empty and 'fault_start_local' in mfs.columns:
    # parse; keep tz-aware in Asia/Shanghai
    mfs['fault_start_dt'] = pd.to_datetime(mfs['fault_start_local'], utc=True).dt.tz_convert(tz)
else:
    mfs['fault_start_dt'] = pd.NaT

# Compute KPI prefix (substring before first '-')
def kpi_prefix(k):
    try:
        return str(k).split('-',1)[0]
    except Exception:
        return str(k)

mfs['kpi_prefix'] = mfs['kpi_name'].apply(kpi_prefix)

# Aggregate metric info per component
agg_metric = mfs[mfs['component'].isin(components)].groupby('component', sort=False).agg(
    total_segments = ('component','size'),
    count_segments_len_ge2 = ('length_in_points', lambda s: int((s>=2).sum())),
    earliest_fault_start_dt = ('fault_start_dt','min'),
    max_breach_ratio = ('max_breach_ratio','max'),
    mean_breach_ratio = ('max_breach_ratio','mean')
).reset_index()

# Format earliest_fault_start_local
agg_metric['earliest_fault_start_local'] = agg_metric['earliest_fault_start_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else pd.NA
)

# Top 3 KPI prefix categories per component
top_prefix_list = []
for comp in components:
    subset = mfs[mfs['component']==comp]
    if subset.empty:
        top_prefix_list.append((comp,""))
        continue
    counts = subset['kpi_prefix'].value_counts().head(3)
    formatted = ", ".join([f"{p}:{int(c)}" for p,c in counts.items()])
    top_prefix_list.append((comp, formatted))

top_prefix_df = pd.DataFrame(top_prefix_list, columns=['component','top_KPI_categories'])

# Merge to ensure all components present
component_full_summary = pd.DataFrame({'component':components}).merge(agg_metric, on='component', how='left').merge(top_prefix_df, on='component', how='left')

# Fill NaNs for components without entries
component_full_summary['total_segments'] = component_full_summary['total_segments'].fillna(0).astype(int)
component_full_summary['count_segments_len_ge2'] = component_full_summary['count_segments_len_ge2'].fillna(0).astype(int)
component_full_summary['max_breach_ratio'] = component_full_summary['max_breach_ratio'].fillna(0.0).astype(float)
component_full_summary['mean_breach_ratio'] = component_full_summary['mean_breach_ratio'].fillna(0.0).astype(float)
component_full_summary['earliest_fault_start_local'] = component_full_summary['earliest_fault_start_local'].fillna(pd.NA)
component_full_summary['top_KPI_categories'] = component_full_summary['top_KPI_categories'].fillna("")

# Now incorporate trace incoming edge metrics from trace_edges
# Ensure trace_edges exists
if 'trace_edges' not in globals():
    trace_edges = pd.DataFrame(columns=['edge_source','edge_target','aggregated_trace_group_count','aggregated_sum_anomalies','earliest_anomaly_epoch','earliest_anomaly_local'])

te = trace_edges.copy()

# Aggregate incoming per target
if not te.empty:
    incoming = te.groupby('edge_target', sort=False).agg(
        trace_incoming_edge_count = ('edge_source','nunique'),
        trace_total_incoming_anomalies = ('aggregated_sum_anomalies','sum'),
        earliest_incoming_epoch = ('earliest_anomaly_epoch','min')
    ).reset_index().rename(columns={'edge_target':'component'})
    incoming['earliest_incoming_local'] = incoming['earliest_incoming_epoch'].apply(
        lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(e) else pd.NA
    )
else:
    incoming = pd.DataFrame(columns=['component','trace_incoming_edge_count','trace_total_incoming_anomalies','earliest_incoming_epoch','earliest_incoming_local'])

# Merge incoming info into component_full_summary
component_full_summary = component_full_summary.merge(incoming[['component','trace_incoming_edge_count','trace_total_incoming_anomalies','earliest_incoming_local']], on='component', how='left')

# Fill missing incoming with zeros/NA
component_full_summary['trace_incoming_edge_count'] = component_full_summary['trace_incoming_edge_count'].fillna(0).astype(int)
component_full_summary['trace_total_incoming_anomalies'] = component_full_summary['trace_total_incoming_anomalies'].fillna(0).astype(int)
component_full_summary['earliest_incoming_local'] = component_full_summary['earliest_incoming_local'].fillna(pd.NA)

# Select and order final compact columns
compact_cols = [
    'component',
    'total_segments',
    'count_segments_len_ge2',
    'earliest_fault_start_local',
    'max_breach_ratio',
    'mean_breach_ratio',
    'top_KPI_categories',
    'trace_incoming_edge_count',
    'trace_total_incoming_anomalies'
]

component_summary_compact = component_full_summary[compact_cols].copy()

# Sort by trace_total_incoming_anomalies desc then total_segments desc
component_summary_compact = component_summary_compact.sort_values(['trace_total_incoming_anomalies','total_segments'], ascending=[False, False]).reset_index(drop=True)

# Store full per-component summary in kernel variable
component_full_summary_table = component_full_summary

# Display the compact table for the 8 components
component_summary_compact

```
Out[9]:
```
Summary (per-component compact view; times UTC+8)

For each component I report: total_segments, segments with length>=2, earliest fault start, max breach ratio, mean breach ratio, top 3 KPI-prefix contributors (prefix:count), trace incoming edge count, and total incoming trace anomalies.

1. MG01
   - total_segments: 76; segments_len>=2: 5
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:68, JVM:8
   - trace_incoming_edge_count: 9; trace_total_incoming_anomalies: 127

2. MG02
   - total_segments: 79; segments_len>=2: 1
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:71, JVM:8
   - trace_incoming_edge_count: 9; trace_total_incoming_anomalies: 122

3. Tomcat02
   - total_segments: 72; segments_len>=2: 4
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:63, Tomcat:9
   - trace_incoming_edge_count: 3; trace_total_incoming_anomalies: 52

4. Tomcat01
   - total_segments: 76; segments_len>=2: 4
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:64, Tomcat:12
   - trace_incoming_edge_count: 3; trace_total_incoming_anomalies: 48

5. Tomcat03
   - total_segments: 74; segments_len>=2: 4
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:62, Tomcat:12
   - trace_incoming_edge_count: 3; trace_total_incoming_anomalies: 44

6. Tomcat04
   - total_segments: 69; segments_len>=2: 6
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:59, Tomcat:10
   - trace_incoming_edge_count: 2; trace_total_incoming_anomalies: 24

7. Mysql02
   - total_segments: 165; segments_len>=2: 4
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: OSLinux:89, Mysql:76
   - trace_incoming_edge_count: 0; trace_total_incoming_anomalies: 0

8. Mysql01
   - total_segments: 153; segments_len>=2: 3
   - earliest_fault_start_local: 2021-03-10T08:30:00+08:00
   - max_breach_ratio: 1.0; mean_breach_ratio: 1.0
   - top KPI prefixes: Mysql:77, OSLinux:76
   - trace_incoming_edge_count: 0; trace_total_incoming_anomalies: 0

Key takeaways
- MG01 and MG02 receive the largest number and severity of incoming anomalous traces (127 and 122 anomalies respectively).
- Tomcat nodes have moderate incoming trace anomalies (Tomcat02/01/03/04 range 24–52 total incoming anomalies).
- Mysql01/Mysql02 show many metric fault segments but no incoming anomalous trace edges (trace_incoming_anomalies = 0).
- Most components show strong breaches (max_breach_ratio = 1.0). Top KPI categories are predominantly OSLinux and JVM/Mysql/Tomcat prefixes.
- The full per-component summary is saved in kernel variable component_full_summary_table, and metric/trace segment tables remain stored in metric_fault_segments and trace_edges.

The original code execution output of IPython Kernel is also provided below for reference:

component  total_segments  count_segments_len_ge2 earliest_fault_start_local  max_breach_ratio  mean_breach_ratio     top_KPI_categories  trace_incoming_edge_count  trace_total_incoming_anomalies
0      MG01              76                       5  2021-03-10T08:30:00+08:00               1.0                1.0      OSLinux:68, JVM:8                          9                             127
1      MG02              79                       1  2021-03-10T08:30:00+08:00               1.0                1.0      OSLinux:71, JVM:8                          9                             122
2  Tomcat02              72                       4  2021-03-10T08:30:00+08:00               1.0                1.0   OSLinux:63, Tomcat:9                          3                              52
3  Tomcat01              76                       4  2021-03-10T08:30:00+08:00               1.0                1.0  OSLinux:64, Tomcat:12                          3                              48
4  Tomcat03              74                       4  2021-03-10T08:30:00+08:00               1.0                1.0  OSLinux:62, Tomcat:12                          3                              44
5  Tomcat04              69                       6  2021-03-10T08:30:00+08:00               1.0                1.0  OSLinux:59, Tomcat:10                          2                              24
6   Mysql02             165                       4  2021-03-10T08:30:00+08:00               1.0                1.0   OSLinux:89, Mysql:76                          0                               0
7   Mysql01             153                       3  2021-03-10T08:30:00+08:00               1.0                1.0   Mysql:77, OSLinux:76                          0                               0```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Ensure incident window epochs exist
if 'start_epoch' not in globals() or 'end_epoch' not in globals():
    start_local = tz.localize(datetime(2021,3,10,8,30,0))
    end_local = tz.localize(datetime(2021,3,10,9,0,0))
    start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
    end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Target component and KPI
comp = 'MG01'
kpi = 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed'

# Full series for global thresholds
series_full = metric[(metric['cmdb_id'] == comp) & (metric['kpi_name'] == kpi)].copy()
series_full['value'] = pd.to_numeric(series_full['value'], errors='coerce')

if series_full.empty:
    global_p95 = pd.NA
    global_p5 = pd.NA
else:
    global_p95 = float(series_full['value'].quantile(0.95))
    global_p5 = float(series_full['value'].quantile(0.05))

# Filter to incident window
mg01_heap_series = series_full[(series_full['timestamp'] >= start_epoch) & (series_full['timestamp'] <= end_epoch)].copy()
mg01_heap_series = mg01_heap_series.sort_values('timestamp').reset_index(drop=True)

# Prepare local timestamp and flags
if not mg01_heap_series.empty:
    mg01_heap_series['timestamp_local'] = pd.to_datetime(mg01_heap_series['timestamp'], unit='s', utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat())
    mg01_heap_series['gt_p95'] = mg01_heap_series['value'] > global_p95
    mg01_heap_series['lt_p5'] = mg01_heap_series['value'] < global_p5
else:
    # ensure the variable exists as empty DataFrame with expected columns
    mg01_heap_series = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','timestamp_local','gt_p95','lt_p5'])

# Earliest anomaly timestamps inside window (local) if any
if not mg01_heap_series.empty and mg01_heap_series['gt_p95'].any():
    earliest_gt_epoch = int(mg01_heap_series.loc[mg01_heap_series['gt_p95'],'timestamp'].min())
    earliest_gt_local = pd.to_datetime(earliest_gt_epoch, unit='s', utc=True).tz_convert(tz).isoformat()
else:
    earliest_gt_local = pd.NA

if not mg01_heap_series.empty and mg01_heap_series['lt_p5'].any():
    earliest_lt_epoch = int(mg01_heap_series.loc[mg01_heap_series['lt_p5'],'timestamp'].min())
    earliest_lt_local = pd.to_datetime(earliest_lt_epoch, unit='s', utc=True).tz_convert(tz).isoformat()
else:
    earliest_lt_local = pd.NA

# Compact display table limited to 31 rows
display_table = mg01_heap_series[['timestamp_local','value','gt_p95','lt_p5']].head(31)

# Store the filtered series in kernel variable mg01_heap_series (already set)
display_table, global_p95, global_p5, earliest_gt_local, earliest_lt_local, mg01_heap_series.shape

```
Out[10]:
```
Summary for MG01 — KPI "JVM-Memory_7779_JVM_Memory_HeapMemoryUsed"

- Global thresholds (computed from the full series):
  - P95 = 1,023,540,048.0
  - P5  =   261,598,656.0

- Incident-window series (2021-03-10 08:30–09:00 UTC+8)
  - Returned a compact table of 31 rows (one per minute) with columns: timestamp_local, value, gt_p95 (value > global P95), lt_p5 (value < global P5).
  - The filtered series is stored in kernel variable mg01_heap_series.

- Earliest anomalies inside the window:
  - Earliest value > P95: 2021-03-10T08:34:00+08:00
  - Earliest value < P5: 2021-03-10T08:43:00+08:00

If you want, I can show the full 31-row table here or give counts of how many points in the window exceeded P95 or were below P5.

The original code execution output of IPython Kernel is also provided below for reference:

(              timestamp_local         value  gt_p95  lt_p5
0   2021-03-10T08:30:00+08:00  7.452599e+08   False  False
1   2021-03-10T08:31:00+08:00  6.623438e+08   False  False
2   2021-03-10T08:32:00+08:00  3.951468e+08   False  False
3   2021-03-10T08:33:00+08:00  3.234484e+08   False  False
4   2021-03-10T08:34:00+08:00  1.023870e+09    True  False
..                        ...           ...     ...    ...
26  2021-03-10T08:56:00+08:00  8.472541e+08   False  False
27  2021-03-10T08:57:00+08:00  3.867354e+08   False  False
28  2021-03-10T08:58:00+08:00  8.539124e+08   False  False
29  2021-03-10T08:59:00+08:00  4.780037e+08   False  False
30  2021-03-10T09:00:00+08:00  1.023210e+09   False  False

[31 rows x 4 columns], 1023540048.0, 261598656.0, '2021-03-10T08:34:00+08:00', '2021-03-10T08:43:00+08:00', (31, 7))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```